# 02 — Position Reconstruction

Goal: from raw Lendle/Init/Agni/Moe event logs, reconstruct per-wallet position state at hourly cadence.

Output: `data/positions.parquet` with schema `(timestamp_hour, wallet, protocol, asset, position_value_usd)`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
from loguru import logger

## 1. Decode raw Lendle Supply/Borrow logs into position deltas

Each Supply event → +position. Each Withdraw → -position. Track per (wallet, reserve) cumulative balance.

In [ ]:
from eth_abi import decode
from eth_utils import to_checksum_address

def decode_supply_log(log: dict) -> dict:
    """Aave V3 IPool Supply(reserve, user, onBehalfOf, amount, referralCode).
    
    Indexed: reserve (topic[1]), onBehalfOf (topic[3]), referralCode (topic[4]?).
    Non-indexed (data): user, amount.
    """
    reserve = '0x' + log['topics'][1][-40:]
    on_behalf_of = '0x' + log['topics'][3][-40:]
    referral = int(log['topics'][4], 16) if len(log['topics']) > 4 else 0
    data = bytes.fromhex(log['data'][2:])
    user, amount = decode(['address', 'uint256'], data)
    return {
        'block_number': int(log['blockNumber'], 16),
        'tx_hash': log['transactionHash'],
        'reserve': to_checksum_address(reserve),
        'user': to_checksum_address(user),
        'on_behalf_of': to_checksum_address(on_behalf_of),
        'amount': amount,
        'referral': referral,
    }

## 2. Resample to hourly position snapshots

Cumulative balance per (wallet, reserve) as a step function; resample to hourly grid.

In [ ]:
def reconstruct_hourly_positions(deltas: pd.DataFrame) -> pd.DataFrame:
    """
    Args:
        deltas: rows of (timestamp, wallet, reserve, amount_delta)
    Returns:
        hourly positions: (timestamp_hour, wallet, reserve, cumulative_amount)
    """
    deltas = deltas.sort_values('timestamp')
    deltas['cumulative_amount'] = (
        deltas
        .groupby(['wallet', 'reserve'])['amount_delta']
        .cumsum()
    )
    # Reindex to hourly
    hourly_grid = pd.DataFrame({
        'timestamp_hour': pd.date_range(
            deltas['timestamp'].min().floor('h'),
            deltas['timestamp'].max().ceil('h'),
            freq='h',
        ),
    })
    # Cross join with (wallet, reserve) pairs and forward-fill cumulative balance
    pairs = deltas[['wallet', 'reserve']].drop_duplicates()
    grid = hourly_grid.merge(pairs, how='cross')
    grid = grid.merge(
        deltas[['timestamp', 'wallet', 'reserve', 'cumulative_amount']],
        left_on=['timestamp_hour', 'wallet', 'reserve'],
        right_on=['timestamp', 'wallet', 'reserve'],
        how='left',
    )
    grid['cumulative_amount'] = (
        grid.sort_values('timestamp_hour')
        .groupby(['wallet', 'reserve'])['cumulative_amount']
        .ffill()
        .fillna(0.0)
    )
    return grid[['timestamp_hour', 'wallet', 'reserve', 'cumulative_amount']]

## 3. Translate amounts to USD via Pyth prices

Pyth on Mantle Mainnet: 0xA2aa501b19aff244D90cc15a4Cf739D2725B5729

For backtest we pull historical Pyth price updates via Pyth Hermes API, not on-chain reads (faster).

In [ ]:
# import httpx
# PYTH_HERMES = 'https://hermes.pyth.network'
# def fetch_historical_price(feed_id: str, timestamp: int) -> float:
#     r = httpx.get(f'{PYTH_HERMES}/v2/updates/price/{timestamp}', params={'ids[]': feed_id})
#     r.raise_for_status()
#     parsed = r.json()['parsed'][0]
#     return float(parsed['price']['price']) * (10 ** parsed['price']['expo'])

## 4. Per-wallet hourly NAV series

Sum positions across protocols and reserves to get one NAV value per (wallet, hour). Feeds into 03_sortino_calc.

In [ ]:
# nav_per_wallet = positions.groupby(['timestamp_hour', 'wallet'])['position_value_usd'].sum().reset_index()
# nav_per_wallet.to_parquet('../data/wallet_nav_hourly.parquet')